# Project 1: Predicting Fatigue-Life Class from Process Parameters

In this notebook, you will build a simple **supervised machine learning** workflow.

**Main question:**  
Can process parameters alone help classify whether a printed part belongs to a **short-life** or **long-life** fatigue group?

You will use a prepared CSV file.


## 1. Connect Google Drive

Run this cell in Google Colab so the notebook can access your dataset from Google Drive.

After mounting Drive, check that your CSV file is inside the correct folder.


In [ ]:
# ============================================================
# Cell 1: Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

## 2. Set the dataset path

Update `csv_path` based on where you saved the project folder in Google Drive.

Dataset file:

`project1_process_parameters_fatigue_class.csv`


In [ ]:
# ============================================================
# Cell 2: Set CSV file path
# ============================================================

# TODO: Change this path if your file is saved in a different Google Drive folder
csv_path = "/content/drive/MyDrive/SUPREME_2026/Project_1_Process_Parameters/data/project1_process_parameters_fatigue_class.csv"

print("CSV path:")
print(csv_path)


## 3. Import libraries

We will use:

- `pandas` and `numpy` for data handling
- `matplotlib` for plots
- `scikit-learn` for machine learning models and evaluation


In [ ]:
# ============================================================
# Cell 3: Import required libraries
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


## 4. Load the dataset

Each row represents one printed fatigue specimen.

The input features are process parameters.  
The target column is `fatigue_life_class`.


In [ ]:
# ============================================================
# Cell 4: Load the CSV dataset
# ============================================================

df = pd.read_csv(csv_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

df.head()


## 5. Quick data exploration

Before training a model, always inspect the dataset.

Things to check:

- How many rows and columns are there?
- What are the column names?
- Are there missing values?
- How many samples are in each class?


In [ ]:
# ============================================================
# Cell 5: Explore the dataset
# ============================================================

# 1: Print all column names

# 2: Check missing values

# 3: Display summary statistics



In [ ]:
# ============================================================
# Cell 6: Check class balance
# ============================================================

# Count how many specimens are short_life and long_life



## 6. Plot class balance

This plot helps us see whether the dataset has more short-life or long-life samples.


In [ ]:
# ============================================================
# Cell 7: Plot class balance
# ============================================================

# Create a bar plot for fatigue_life_class



## 7. Select input features and target

For supervised learning, we need:

- `X`: input features
- `y`: target label

Do **not** use `part_id` as a model input. It is only an identifier.


In [ ]:
# ============================================================
# Cell 8: Select features and target
# ============================================================

# Fill in the feature columns
feature_cols = [
    # "power",
    # "scan_speed",
    # "hatch_spacing",
    # "layer_thickness",
    # "ved"
]

target_col = "fatigue_life_class"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

X.head()


## 8. Encode the target labels

Most machine learning models need numerical labels.

For example:

- `long_life` may become 0
- `short_life` may become 1

The exact mapping depends on alphabetical order, so we print the mapping.


In [ ]:
# ============================================================
# Cell 9: Encode target labels
# ============================================================

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Class mapping:")
for class_name, encoded_value in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(class_name, "->", encoded_value)


## 9. Split the data into training and testing sets

Training data is used to teach the model.  
Testing data is used to evaluate the model on unseen examples.

We use `stratify=y_encoded` so both classes are represented in train and test sets.


In [ ]:
# ============================================================
# Cell 10: Train-test split
# ============================================================

# TODO: Complete the train_test_split call
X_train, X_test, y_train, y_test = train_test_split(
    # X,
    # y_encoded,
    # test_size=0.25,
    # random_state=42,
    # stratify=y_encoded
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)


## 10. Define supervised learning models

You will compare several models. Some are simple, and some are stronger ensemble models.

Suggested models:

- Decision Tree
- Random Forest
- SVM
- Bagging Classifier
- XGBoost, if available


In [ ]:
# ============================================================
# Cell 11: Optional XGBoost import
# ============================================================

# XGBoost may not always be installed.
# In Colab, you can usually install it with: !pip install xgboost

try:
    from xgboost import XGBClassifier
    xgboost_available = True
    print("XGBoost is available.")
except Exception as e:
    xgboost_available = False
    print("XGBoost is not available. We will skip it.")
    print("Error:", e)


In [ ]:
# ============================================================
# Cell 12: Define models
# ============================================================

models = {}

#  Add a Decision Tree model
# models["Decision Tree"] = ...

#  Add a Random Forest model
# models["Random Forest"] = ...

# SVM needs feature scaling, so we use a Pipeline
# models["SVM"] = Pipeline([
#     ("scaler", StandardScaler()),
#     ("svm", SVC(kernel="rbf", probability=True, random_state=42))
# ])

#  Add a Bagging Classifier
# models["Bagging"] = ...

# Optional XGBoost model
if xgboost_available:
    models["XGBoost"] = XGBClassifier(
        n_estimators=50,
        max_depth=2,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    )

print("Models prepared:")
for name in models.keys():
    print("-", name)


## 11. Train and evaluate the models

This section is mostly provided for you.

For each model, we will:

1. Train the model using the training data
2. Predict the testing data
3. Calculate accuracy
4. Print a classification report
5. Plot a confusion matrix

The confusion matrix is very useful because it shows which class the model is missing.


In [ ]:
# ============================================================
# Cell 13: Train and evaluate models on test set
# Confusion matrix plots use font size 20
# ============================================================

test_results = []

for model_name, model in models.items():

    print("=" * 60)
    print("Model:", model_name)

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred)

    print("Test Accuracy:", acc)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    ))

    # Plot confusion matrix
    fig, ax = plt.subplots(figsize=(7, 6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=label_encoder.classes_
    )

    disp.plot(
        cmap="Blues",
        values_format="d",
        ax=ax,
        colorbar=False
    )

    ax.set_title(f"{model_name} Confusion Matrix", fontsize=20)
    ax.set_xlabel("Predicted Label", fontsize=20)
    ax.set_ylabel("True Label", fontsize=20)

    ax.tick_params(axis="x", labelsize=20, rotation=30)
    ax.tick_params(axis="y", labelsize=20)

    for text in ax.texts:
        text.set_fontsize(20)

    plt.tight_layout()
    plt.show()

    test_results.append({
        "model": model_name,
        "test_accuracy": acc
    })

test_results_df = pd.DataFrame(test_results).sort_values(
    "test_accuracy",
    ascending=False
)

test_results_df


## 12. Cross-validation

Because the dataset is small, one train/test split may be unstable.

Cross-validation gives a more reliable estimate by repeating training/testing on different splits.


In [ ]:
# ============================================================
# Cell 14: Cross-validation model comparison
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = []

for model_name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y_encoded,
        cv=cv,
        scoring="accuracy"
    )

    cv_results.append({
        "model": model_name,
        "cv_mean_accuracy": scores.mean(),
        "cv_std_accuracy": scores.std(),
        "all_cv_scores": scores
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(
    "cv_mean_accuracy",
    ascending=False
)

cv_results_df


## 13. Voting ensemble

A voting ensemble combines multiple models.

The goal is to see whether combining models gives better performance than using one model alone.


In [ ]:
# ============================================================
# Cell 15: Voting ensemble model
# ============================================================

voting_estimators = [
    ("dt", DecisionTreeClassifier(max_depth=3, random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)),
    ("svm", Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", probability=True, random_state=42))
    ]))
]

if xgboost_available:
    voting_estimators.append(
        ("xgb", XGBClassifier(
            n_estimators=50,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42
        ))
    )

voting_model = VotingClassifier(
    estimators=voting_estimators,
    voting="soft"
)

voting_model.fit(X_train, y_train)

y_pred_voting = voting_model.predict(X_test)

print("Voting Ensemble Test Accuracy:", accuracy_score(y_test, y_pred_voting))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_voting))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_voting,
    target_names=label_encoder.classes_,
    zero_division=0
))


## 14. Feature importance

Random Forest can give a simple estimate of which features were used most.

Be careful: feature importance does **not** prove a physical cause. It only tells us what the model used for prediction.


In [ ]:
# ============================================================
# Cell 16: Random Forest feature importance
# ============================================================

# Train a Random Forest model using all data

# Create a dataframe with feature names and importance values

# Hint:
# rf_model.feature_importances_


In [ ]:
# ============================================================
# Cell 17: Plot feature importance
# ============================================================

# Plot feature importance using a horizontal bar plot



## 15. Final discussion questions

Answer these in your own words:

1. Which model gave the best test accuracy?
2. Which model gave the best cross-validation accuracy?
3. Did the model predict both classes well?
4. Which class was harder to predict?
5. Why might process parameters alone be limited for fatigue-life prediction?
6. What other features could improve the model?


## Key takeaway

This project is not about getting perfect accuracy.

The main lesson is:

**Machine learning performance depends strongly on the information contained in the input features.**

If many parts have similar process parameters but different fatigue lives, process parameters alone may not be enough for reliable fatigue-life prediction.
